# MathLens Companion: Fractions & Decimals Tutor Bot (RAG)

This is a RAG chatbot built as **teacher material** for
secondary-school students who are behind on **fractions and decimals**. A student can ask it
plain-language questions and get a short, step-by-step explanation grounded in real reference
text.

---
## 1. Installations and Settings 🛠️

In [ ]:
!pip install -q jedi
!pip install -q llama-index-core
!pip install -q llama-index-llms-groq
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

---
## 2. Setting up the LLM



In [ ]:
from llama_index.llms.groq import Groq

model = "openai/gpt-oss-120b"

llm = Groq(model=model)

---
## 3. RAG





#3.1. Loading the materials (PDF uploaded)
## Files--content/data


### 3.2. Load the data

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("/content/data").load_data()
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print("-", doc.metadata.get("file_name"))

Loaded 19 documents
- 1_Fractions-decimals-and-percentages.pdf
- 1_Fractions-decimals-and-percentages.pdf
- 1_Fractions-decimals-and-percentages.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf
- 2_fractions.pdf


### 3.3. Splitting the documents



In [ ]:
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

text_splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

docs = text_splitter.get_nodes_from_documents(documents)
print(f"Documents split into {len(docs)} chunks")
docs[0]

Documents split into 19 chunks


TextNode(id_='7a12cee4-ceb5-4739-8524-107b905261b8', embedding=None, metadata={'page_label': '1', 'file_name': '1_Fractions-decimals-and-percentages.pdf', 'file_path': '/content/data/1_Fractions-decimals-and-percentages.pdf', 'file_type': 'application/pdf', 'file_size': 130825, 'creation_date': '2026-08-31', 'last_modified_date': '2026-08-31'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='f7cf1253-c37a-4c4f-90e6-2ef21c3c4fc2', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': '1_Fractions-decimals-and-percentages.pdf', 'file_path': '/content/data/1_Fractions-decimals-and-percentages.pdf', 'file_type': 'application/pdf', 'file_size': 130825, 'creation_date': '2026-08-31', 'la

### 3.4. Creating vectors with embeddings

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings_folder = "/content/embedding_model/"

embeddings = HuggingFaceEmbedding(
    model_name=embedding_model, cache_folder=embeddings_folder
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
test_text = "A fraction like 3/4 has a numerator on top and a denominator on the bottom."
query_result = embeddings.get_text_embedding(test_text)

characters = len(test_text)
dimensions = len(query_result)
print(f"The {characters} character sentence was transformed into a {dimensions} dimension vector")

The 75 character sentence was transformed into a 384 dimension vector


### 3.5. Creating a vector database

In [ ]:
from llama_index.core import VectorStoreIndex

documents = SimpleDirectoryReader("/content/data").load_data()

vector_index = VectorStoreIndex.from_documents(
    documents, transformations=[text_splitter], embed_model=embeddings
)

In [ ]:
vector_index.storage_context.persist(persist_dir="/content/vector_index")

Quick check: which chunks come back for a student-style question, before adding the LLM?

In [ ]:
retriever = vector_index.as_retriever(similarity_top_k=2)
retriever.retrieve("How do I turn a fraction into a decimal?")

[NodeWithScore(node=TextNode(id_='2d001aeb-9b40-4a95-809e-dbdd8d41f150', embedding=None, metadata={'page_label': '10', 'file_name': '2_fractions.pdf', 'file_path': '/content/data/2_fractions.pdf', 'file_type': 'application/pdf', 'file_size': 157728, 'creation_date': '2026-08-31', 'last_modified_date': '2026-08-31'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='c14ddf34-e03e-4547-a9de-82b370afe471', node_type='4', metadata={'page_label': '10', 'file_name': '2_fractions.pdf', 'file_path': '/content/data/2_fractions.pdf', 'file_type': 'application/pdf', 'file_size': 157728, 'creation_date': '2026-08-31', 'last_modified_date': '2026-08-31'}, hash='2f3b75023fb5a58ae95fda1a9ce882d4628e6d2305c65a4015b8e607d10c5

### 3.6. Adding a tutoring-style prompt

****not just answer — short, encouraging, step-by-step, in language appropriate for a
secondary-school student who's finding this topic hard, and to say clearly when the answer isn't
in the material rather than guessing.

In [ ]:
from llama_index.core.prompts import PromptTemplate

input_template = """You are a patient, encouraging math tutor helping a secondary-school
student who is finding this topic difficult. Use only the context below to answer -- do not use
outside knowledge, and if the context doesn't contain the answer, say so honestly instead of
guessing.

Explain the answer in simple language, step by step where it helps, using a short example if one
fits. Keep the whole answer under 100 words.

Context:
{context_str}

Student's question: {query_str}
Tutor's answer:"""

prompt = PromptTemplate(template=input_template)

### 3.7. RAG — chaining it all together

In [ ]:
query_engine = vector_index.as_query_engine(
    llm=llm, text_qa_template=prompt, similarity_top_k=2, response_mode="tree_summarize"
)

In [ ]:
questions = [
    "What is a numerator and what is a denominator?",
    "How do I convert a fraction like 3/4 into a decimal?",
    "How do I round a decimal number to two decimal places?",
    "What is a mixed number and how do I turn it into an improper fraction?",
]

for q in questions:
    answer = query_engine.query(q)
    print(f"Q: {q}\nA: {answer}\n{'-'*60}")

Q: What is a numerator and what is a denominator?
A: A numerator is the top number in a fraction; it tells how many parts are being considered.  
A denominator is the bottom number in a fraction; it tells into how many equal parts the whole unit is divided.
------------------------------------------------------------
Q: How do I convert a fraction like 3/4 into a decimal?
A: To change a common fraction to a decimal, divide the numerator by the denominator.

\( \displaystyle \frac{3}{4}=3 \div 4 = 0.75 \)

So \( \frac{3}{4} \) expressed as a decimal is **0.75**.
------------------------------------------------------------
Q: How do I round a decimal number to two decimal places?
A: To round a number to two decimal places, keep only the digits in the hundredths position and decide whether to adjust the last kept digit based on the next digit (the thousandths place).

1. **Locate the hundredths digit** – this is the second digit to the right of the decimal point.  
2. **Look at the digit 

#### 3.7.1. Exploring a returned answer

In [ ]:
answer = query_engine.query("Why do we simplify fractions to their lowest terms?")
print(answer.response)

Simplifying a fraction removes any common factors that appear in both the numerator and the denominator. When those shared factors are cancelled, the fraction is expressed in its most compact form, which makes the numbers easier to read, compare, and work with in further calculations such as addition, subtraction, multiplication, or division. A reduced‑to‑lowest‑terms fraction also clearly shows the true size of the part relative to the whole, without extra “extra” multiples that do not change its value. In short, simplifying yields a clearer, more efficient representation of the same quantity.


In [ ]:
# which chunks (and which source page) were used to generate this answer?
for node in answer.source_nodes:
    print(node.metadata.get("file_name"), "| score:", round(node.score, 3))
    print(node.text[:200].replace("\n", " "), "...")
    print("-" * 60)

2_fractions.pdf | score: 0.553
MATH 6007  2  INTRODUCTION    Fractions are represented by an integer above a line with another integer below  the line.  This is known as a common fraction.  The top number is called the  numerator a ...
------------------------------------------------------------
2_fractions.pdf | score: 0.536
MATH 6007  7  Subtraction of Fractions    The same principles apply in subtraction as in addition; the quantities are brought  to the same denominator and then one of the numerators is subtracted from ...
------------------------------------------------------------


---
## 4. Experimenting with parameters



### 4.1. Varying `chunk_size`

In [ ]:
test_question = "What's the difference between a proper fraction and an improper fraction?"

for size, overlap in [(300, 50), (800, 150), (1500, 200)]:
    splitter = SentenceSplitter(chunk_size=size, chunk_overlap=overlap)
    idx = VectorStoreIndex.from_documents(
        documents, transformations=[splitter], embed_model=embeddings
    )
    qe = idx.as_query_engine(
        llm=llm, text_qa_template=prompt, similarity_top_k=2, response_mode="tree_summarize"
    )
    answer = qe.query(test_question)
    print(f"chunk_size={size}, overlap={overlap}\n-> {answer}\n{'='*60}")

chunk_size=300, overlap=50
-> A proper fraction has a numerator that is smaller than its denominator, so its value is less than one. An improper fraction has a numerator that is equal to or larger than its denominator, giving a value of one or more.
chunk_size=800, overlap=150
-> A proper fraction is a fraction whose numerator is smaller than its denominator, so its value is less than one. An improper fraction has a numerator that is equal to or larger than its denominator, giving a value that is one or greater.
chunk_size=1500, overlap=200
-> A proper fraction is a fraction whose numerator is smaller than its denominator, so its value is less than one. An improper fraction has a numerator that is equal to or larger than its denominator, giving a value that is one or greater.


### 4.2. Varying `similarity_top_k`

In [ ]:
base_index = VectorStoreIndex.from_documents(
    documents, transformations=[SentenceSplitter(chunk_size=800, chunk_overlap=150)],
    embed_model=embeddings,
)

for k in [1, 2, 5]:
    qe = base_index.as_query_engine(
        llm=llm, text_qa_template=prompt, similarity_top_k=k, response_mode="tree_summarize"
    )
    answer = qe.query(test_question)
    print(f"top_k={k}\n-> {answer}\n{'='*60}")

top_k=1
-> A proper fraction is one whose numerator is smaller than its denominator, so its value is less than 1. An improper fraction has a numerator that is equal to or larger than its denominator, giving a value of 1 or greater.
top_k=2
-> A proper fraction is a fraction whose numerator is smaller than its denominator, so its value is less than one. An improper fraction has a numerator that is equal to or larger than its denominator, giving a value that is one or greater.
top_k=5
-> A proper fraction represents a quantity smaller than one; its numerator is smaller than its denominator (e.g., 2/3, 3/8). An improper fraction represents a quantity equal to or greater than one; its numerator is larger than—or equal to—its denominator (e.g., 5/3, 9/8).


### 5. Live Q&A Demo

Ask the tutor bot your own questions about fractions and decimals. Type `end` to stop.

In [ ]:
# Start the conversation loop
while True:
    user_input = input("You: ")

    # Check for exit condition
    if user_input.lower() == "end":
        print("Ending the conversation. Goodbye!")
        break

    # Get the response from your existing query engine
    response = query_engine.query(user_input)

    # Print the tutor bot's response
    print("Tutor Bot:", response.response)

You: Convert 3/4 into percentage
Tutor Bot: 75%
You: what is improper fraction?
Tutor Bot: An improper fraction is a fraction whose numerator is equal to or larger than its denominator, so the value of the fraction is one or greater. Because the whole‑number part can be taken out, an improper fraction can always be rewritten as a mixed number (a whole number plus a proper fraction).
You: end
Ending the conversation. Goodbye!
